# Heart Disease Detection — Champion Model: XGBoost Classifier (CRISP-DM Stage 2)

This notebook implements a **Tree-based Ensemble model using XGBoost** as a candidate champion model for heart disease prediction.

## Stage 2 Objectives
- Build a scikit-learn **Pipeline** (scaling + XGBClassifier) to prevent data leakage
- Handle class imbalance via `scale_pos_weight`
- Tune hyperparameters with **RandomizedSearchCV** (50 iterations, 5-fold CV)
- Output **probability scores** (0.0–1.0) and apply a custom **decision threshold**
- Evaluate with Classification Report, Confusion Matrix, ROC-AUC
- Perform **failure analysis** (top 10 most confident failures)
- Produce comparison metrics for group integration

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    f1_score,
    accuracy_score,
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully!")
print(f"Random State: {RANDOM_STATE}")

## 2. Load Data from Data Preparation Stage

In [ ]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").values.ravel()
y_test = pd.read_csv("y_test.csv").values.ravel()

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test  shape: {y_test.shape}")

## 3. Encode Target Variable & Check Class Imbalance

In [ ]:
# Encode target: "Yes" -> 1, "No" -> 0
le = LabelEncoder()
le.fit(["No", "Yes"])
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"\nTraining set target distribution:")
unique, counts = np.unique(y_train_enc, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {le.inverse_transform([label])[0]} ({label}): {count} ({count/len(y_train_enc)*100:.2f}%)")

# Calculate scale_pos_weight for imbalance handling
neg_count = np.sum(y_train_enc == 0)
pos_count = np.sum(y_train_enc == 1)
scale_pos_weight = neg_count / pos_count
print(f"\nClass imbalance ratio (No:Yes): {scale_pos_weight:.2f}:1")
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

## 4. Build Scikit-learn Pipeline

The pipeline bundles a `StandardScaler` and the `XGBClassifier` together to prevent data leakage — scaling statistics are computed only from training folds during cross-validation.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("xgb", XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=2,
    )),
])

print("Pipeline created:")
print(pipeline)

## 5. Hyperparameter Tuning with RandomizedSearchCV

We tune `n_estimators`, `max_depth`, and `learning_rate` over **50 iterations** with **5-fold stratified CV**.

In [ ]:
param_distributions = {
    "xgb__n_estimators": [100, 200, 300, 400, 500],
    "xgb__max_depth": [3, 4, 5, 6, 7, 8, 10],
    "xgb__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2, 0.3],
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1",
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=2,
    verbose=1,
    return_train_score=True,
)

print("Starting RandomizedSearchCV (50 iterations, 5-fold CV)...")
random_search.fit(X_train, y_train_enc)
print("\nSearch complete!")

In [ ]:
print("Best hyperparameters found:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV F1-score: {random_search.best_score_:.4f}")

# Extract best model from the search
best_pipeline = random_search.best_estimator_

## 6. Probability Output & Custom Decision Threshold

Instead of the default 0.5 threshold, we apply a **custom threshold of 0.65** as an actionable business policy — a higher threshold means we only flag patients as at-risk when the model is more confident, reducing false positives in a screening context.

In [ ]:
# Predict probabilities on the test set
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print("Probability score summary (P(HeartAttack=Yes)):")
print(f"  Min:    {y_proba.min():.4f}")
print(f"  Max:    {y_proba.max():.4f}")
print(f"  Mean:   {y_proba.mean():.4f}")
print(f"  Median: {np.median(y_proba):.4f}")

# Apply custom decision threshold
THRESHOLD = 0.65
y_pred_custom = (y_proba >= THRESHOLD).astype(int)

# Also get default threshold predictions for comparison
y_pred_default = (y_proba >= 0.5).astype(int)

print(f"\nPredictions with default threshold (0.50):")
print(f"  Predicted Yes: {np.sum(y_pred_default == 1)}, Predicted No: {np.sum(y_pred_default == 0)}")
print(f"\nPredictions with custom threshold ({THRESHOLD}):")
print(f"  Predicted Yes: {np.sum(y_pred_custom == 1)}, Predicted No: {np.sum(y_pred_custom == 0)}")

In [ ]:
# Visualise the probability distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_proba[y_test_enc == 0], bins=50, alpha=0.6, label="Actual: No", color="steelblue")
ax.hist(y_proba[y_test_enc == 1], bins=50, alpha=0.6, label="Actual: Yes", color="salmon")
ax.axvline(THRESHOLD, color="red", linestyle="--", linewidth=2, label=f"Threshold = {THRESHOLD}")
ax.set_xlabel("Predicted Probability of Heart Disease")
ax.set_ylabel("Count")
ax.set_title("Distribution of Predicted Probabilities")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Evaluation & Failure Analysis

### 7.1 Classification Report (Custom Threshold)

In [ ]:
target_names = ["No Heart Disease (0)", "Heart Disease (1)"]
print(f"Classification Report (Threshold = {THRESHOLD})")
print("=" * 60)
print(classification_report(y_test_enc, y_pred_custom, target_names=target_names))

### 7.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Default threshold
ConfusionMatrixDisplay.from_predictions(
    y_test_enc, y_pred_default,
    display_labels=target_names,
    cmap="Blues",
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix (Default Threshold = 0.50)")

# Custom threshold
ConfusionMatrixDisplay.from_predictions(
    y_test_enc, y_pred_custom,
    display_labels=target_names,
    cmap="Blues",
    ax=axes[1],
)
axes[1].set_title(f"Confusion Matrix (Custom Threshold = {THRESHOLD})")

plt.tight_layout()
plt.show()

### 7.3 ROC-AUC Score & Curve

In [ ]:
roc_auc = roc_auc_score(y_test_enc, y_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

fpr, tpr, thresholds = roc_curve(y_test_enc, y_proba)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="steelblue", linewidth=2, label=f"XGBoost (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="Random Classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — XGBoost Classifier")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 7.4 Failure Analysis — Top 10 Most Confident Failures

These are cases where the model was **most confidently wrong**:
- **False Positives:** high predicted probability, but actual label is No (0)
- **False Negatives:** low predicted probability, but actual label is Yes (1)

Analysing these helps us understand the model's mechanical weaknesses.

In [ ]:
# Build a failure analysis dataframe
failure_df = pd.DataFrame({
    "actual": y_test_enc,
    "predicted_prob": y_proba,
    "predicted_label": y_pred_custom,
})
failure_df["error"] = failure_df["actual"] != failure_df["predicted_label"]
failure_df["confidence"] = np.abs(failure_df["predicted_prob"] - 0.5)

# Filter to errors only, sorted by confidence (most confident mistakes first)
errors = failure_df[failure_df["error"]].sort_values("confidence", ascending=False)

print(f"Total misclassifications (threshold={THRESHOLD}): {len(errors)} / {len(failure_df)} ({len(errors)/len(failure_df)*100:.2f}%)")
print(f"\n{'='*80}")
print("TOP 10 MOST CONFIDENT FAILURES")
print(f"{'='*80}")

top_10 = errors.head(10).copy()
top_10["actual_label"] = le.inverse_transform(top_10["actual"])
top_10["failure_type"] = top_10.apply(
    lambda r: "FALSE POSITIVE (predicted Yes, actual No)" if r["actual"] == 0
    else "FALSE NEGATIVE (predicted No, actual Yes)",
    axis=1
)

for i, (idx, row) in enumerate(top_10.iterrows(), 1):
    print(f"\n#{i} | Test index: {idx}")
    print(f"   Predicted probability: {row['predicted_prob']:.4f}")
    print(f"   Actual label:          {row['actual_label']}")
    print(f"   Type:                  {row['failure_type']}")

In [ ]:
# Examine feature values of the top 10 failures
top_10_indices = top_10.index.tolist()
top_10_features = X_test.iloc[top_10_indices].copy()
top_10_features.insert(0, "actual_label", top_10["actual_label"].values)
top_10_features.insert(1, "predicted_prob", top_10["predicted_prob"].values)
top_10_features.insert(2, "failure_type", top_10["failure_type"].values)

print("Feature values for top 10 most confident failures:")
top_10_features.T

## 8. Feature Importance

In [ ]:
# Extract the XGBClassifier from the pipeline
xgb_model = best_pipeline.named_steps["xgb"]
importances = xgb_model.feature_importances_
feature_names = X_train.columns

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.head(20).plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Feature Importance (Gain)")
ax.set_title("Top 20 Feature Importances — XGBoost")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Comparison Metrics Summary

Summary table for comparison against groupmates' models.

In [ ]:
# Cross-validation metrics from the search
cv_results = pd.DataFrame(random_search.cv_results_)
best_idx = random_search.best_index_

mean_cv_f1 = cv_results.loc[best_idx, "mean_test_score"]
std_cv_f1 = cv_results.loc[best_idx, "std_test_score"]

# Test set metrics with custom threshold
test_accuracy = accuracy_score(y_test_enc, y_pred_custom)
test_f1_macro = f1_score(y_test_enc, y_pred_custom, average="macro")
test_f1_yes = f1_score(y_test_enc, y_pred_custom, average="binary")

summary = pd.DataFrame({
    "Model": ["XGBoost (tuned)"],
    "Best CV F1": [f"{mean_cv_f1:.4f} ± {std_cv_f1:.4f}"],
    "Test Accuracy": [f"{test_accuracy:.4f}"],
    "Test F1 (Macro)": [f"{test_f1_macro:.4f}"],
    "Test F1 (Yes)": [f"{test_f1_yes:.4f}"],
    "ROC-AUC": [f"{roc_auc:.4f}"],
    "Threshold": [THRESHOLD],
    "Best Params": [str(random_search.best_params_)],
})

print("=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
summary.T